In [1]:
import sys
import logging
from logging import info

logging.basicConfig(
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO, force=True,
    handlers=[logging.StreamHandler(sys.stdout)])


RADICALE_HOST = "http://vera-webdav-svc.vera.svc.cluster.local:5232"
NEW_USERNAME = "test"

from uuid import uuid4
from aloedav.client import AloeDAVClient
from aloedav.collection.remote_collection import RemoteCollection

from aloedav.exceptions import PreconditionFailed, WebDAVError
from aloedav.collection.local_collection import LocalCollection
from aloedav.model.m00_constant import TodoStatus
from aloedav.model.m02_vcard import VCARD
from aloedav.model.m02_vevent import VEVENT
from aloedav.model.m02_vtodo import VTODO

aloedav_client = AloeDAVClient(RADICALE_HOST, NEW_USERNAME, "", 5, allow_delete_collection=True)
remote_collections = aloedav_client.list_collections()

In [16]:
from aloedav.model.m02_vcard import VCARD
ADDRESSBOOK_ID = f"addressbook-{uuid4()}"

address_book = RemoteCollection.get_or_create_addressbook(
    client=aloedav_client,
    display_name="Test Contacts",
    description="A verified addressbook",
    addressbook_id=ADDRESSBOOK_ID)

initial_sync_token = address_book.refresh_sync_token()
print(initial_sync_token)

print("# =====================")
print("# Insert Contact")
print("# =====================")
contact_filename, insert_etag = address_book.insert_item(contact)
contact_entry = address_book.get_item(contact_filename)

print(f"contact_filename: {contact_filename}\n")
print(f"insert_etag: {insert_etag}\n")
print(f"contact_entry: {contact_entry}\n")

after_insert_sync_obj = address_book.sync_collection(initial_sync_token)
after_insert_sync_token = after_insert_sync_obj["sync_token"]
print("# =====================")
print("# Update Contact")
print("# =====================")
contact_entry.given_name = "UpdatedName"
updated_etag = address_book.update_item(contact_filename, contact_entry, insert_etag)
updated_entry = address_book.get_item(contact_filename)
after_update_sync_obj = address_book.sync_collection(after_insert_sync_token)
after_update_sync_token = after_update_sync_obj["sync_token"]

print(f"updated_etag: {updated_etag}\n")
print(f"updated_entry: {updated_entry}\n")
print(f"after_update_sync_obj: {after_update_sync_obj}\n")
print(f"after_update_sync_token: {after_update_sync_token}\n")

print("# =====================")
print("# Delete Contact")
print("# =====================")
address_book.delete_item(contact_filename, updated_etag)

after_delete_sync_obj = address_book.sync_collection(after_update_sync_token)
after_delete_sync_token = after_delete_sync_obj["sync_token"]

print(f"after_delete_sync_obj: {after_delete_sync_obj}\n")
print(f"after_delete_sync_token: {after_delete_sync_token}\n")

address_book.delete_collection()

2026-01-15 05:08:07,642 - INFO - [AloeDAVClient._create] Collection created: http://vera-webdav-svc.vera.svc.cluster.local:5232/test/addressbook-e45ed779-fedb-41d6-9195-17a578e8317f
http://radicale.org/ns/sync/e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
# =====================
# Insert Contact
# =====================
contact_filename: contact-ac3c031f-9843-4efd-a93f-6124f31188ab.vcf

insert_etag: fc34f5172d516e28ecc6fc70f21de70b35756bfdb729f274de35810183d91419

contact_entry: uid='contact-ac3c031f-9843-4efd-a93f-6124f31188ab' content_type='text/vcard; charset=utf-8' file_ext='vcf' version='3.0' rev='2026-01-15T05:08:07Z' prod_id='-//aloecraft.org//AloeDAV 1.0//EN' extended_attributes={} categories=[] etag='"fc34f5172d516e28ecc6fc70f21de70b35756bfdb729f274de35810183d91419"' raw_contents='BEGIN:VCARD\r\nVERSION:3.0\r\nPRODID:-//aloecraft.org//AloeDAV 1.0//EN\r\nUID:contact-ac3c031f-9843-4efd-a93f-6124f31188ab\r\nEMAIL;TYPE=INTERNET:delete.me@example.com\r\nFN:Delete 

True

In [12]:
from datetime import datetime

CALENDAR_ID = f"calendar-{uuid4()}"
calendar = RemoteCollection.get_or_create_calendar(
    client=aloedav_client,
    display_name="Test Calendar", 
    description="A verified calendar", 
    calendar_id=CALENDAR_ID)

initial_sync_token = calendar.refresh_sync_token()
print(initial_sync_token)

event = VEVENT(
    uid=f"event-{uuid4()}",
    summary="Temporary Event",
    dtstart=datetime(2026, 1, 10, 12, 0, 0),
    dtend=datetime(2026, 1, 10, 13, 0, 0)
)

print("# =====================")
print("# Insert Event")
print("# =====================")
event_filename, insert_etag = calendar.insert_item(event)
event_entry = calendar.get_item(event_filename)

after_insert_sync_obj = calendar.sync_collection(initial_sync_token)
after_insert_sync_token = after_insert_sync_obj["sync_token"]

print(f"event_filename: {event_filename}\n")
print(f"insert_etag: {insert_etag}\n")
print(f"after_insert_sync_obj: {after_insert_sync_obj}\n")
print(f"after_insert_sync_token: {after_insert_sync_token}\n")

print("# =====================")
print("# Update Event")
print("# =====================")
event_entry.summary = "Updated Summary"

updated_etag = calendar.update_item(event_filename, event_entry, insert_etag)
updated_entry = calendar.get_item(event_filename)
after_update_sync_obj = calendar.sync_collection(after_insert_sync_token)
after_update_sync_token = after_update_sync_obj["sync_token"]

print(f"updated_etag: {updated_etag}\n")
print(f"updated_entry: {updated_entry}\n")
print(f"after_update_sync_obj: {after_update_sync_obj}\n")
print(f"after_update_sync_token: {after_update_sync_token}\n")

print("# =====================")
print("# Delete Event")
print("# =====================")
calendar.delete_item(event_filename, updated_etag)

after_delete_sync_obj = calendar.sync_collection(after_update_sync_token)
after_delete_sync_token = after_delete_sync_obj["sync_token"]

print(f"{after_delete_sync_obj}")
print(f"{after_delete_sync_token}")

calendar.delete_collection()

2026-01-15 05:04:42,812 - INFO - [AloeDAVClient._create] Collection created: http://vera-webdav-svc.vera.svc.cluster.local:5232/test/calendar-0e4c0077-6163-4305-a7d0-8d9de22a1f93
http://radicale.org/ns/sync/e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
# =====================
# Insert Event
# =====================
event_filename: event-36acc599-75f4-4c99-b8b9-cfef3ebe78e0.ics

insert_etag: 18bb7bdc8220b5622046ee406380c3a47c5979a6acb3f3363044cc6f70151ba2

after_insert_sync_obj: {'deleted': [], 'updated': [{'href': 'test/calendar-0e4c0077-6163-4305-a7d0-8d9de22a1f93/event-36acc599-75f4-4c99-b8b9-cfef3ebe78e0.ics', 'etag': '18bb7bdc8220b5622046ee406380c3a47c5979a6acb3f3363044cc6f70151ba2', 'uid': 'event-36acc599-75f4-4c99-b8b9-cfef3ebe78e0', 'file_name': 'event-36acc599-75f4-4c99-b8b9-cfef3ebe78e0.ics'}], 'sync_token': 'http://radicale.org/ns/sync/c84ba56cb3bcef58b61632762f2d5d300013b508198bd93363f3582e5e032374'}

after_insert_sync_token: http://radicale.org/ns/sync/c84

True

In [11]:
from uuid import uuid4
from datetime import datetime

CALENDAR_ID = f"calendar-{uuid4()}"
calendar = RemoteCollection.get_or_create_calendar(
    client=aloedav_client,
    display_name="Test Calendar", 
    description="A verified calendar", 
    calendar_id=CALENDAR_ID)

initial_sync_token = calendar.refresh_sync_token()
print(initial_sync_token)

task = VTODO(
    uid=f"task-{uuid4()}",
    summary="Temporary Task",
    dtstart=datetime(2026, 1, 10, 12, 0, 0),
    status=TodoStatus.NEEDS_ACTION
)

print("# =====================")
print("# Insert Task")
print("# =====================")
task_filename, insert_etag = calendar.insert_item(task)
task_entry = calendar.get_item(task_filename)
after_insert_sync_obj = calendar.sync_collection(initial_sync_token)
after_insert_sync_token = after_insert_sync_obj["sync_token"]

print(f"task_filename: {task_filename}\n")
print(f"task_entry: {task_entry}\n")
print(f"after_insert_sync_obj: {after_insert_sync_obj}\n")
print(f"after_insert_sync_token: {after_insert_sync_token}\n")

print("# =====================")
print("# Update Task")
print("# =====================")
task_entry.summary = "Updated Summary"

updated_etag = calendar.update_item(task_filename, task_entry, insert_etag)
updated_entry = calendar.get_item(task_filename)
after_update_sync_obj = calendar.sync_collection(after_insert_sync_token)
after_update_sync_token = after_update_sync_obj["sync_token"]

print(f"updated_etag: {updated_etag}\n")
print(f"updated_entry: {updated_entry}\n")
print(f"after_update_sync_obj: {after_update_sync_obj}\n")
print(f"after_update_sync_token: {after_update_sync_token}\n")

print("# =====================")
print("# Delete Task")
print("# =====================")
calendar.delete_item(task_filename, updated_etag)

after_delete_sync_obj = calendar.sync_collection(after_update_sync_token)
after_delete_sync_token = after_delete_sync_obj["sync_token"]

print(f"after_delete_sync_obj: {after_delete_sync_obj}\n")
print(f"after_delete_sync_token: {after_delete_sync_token}\n")

calendar.delete_collection()

2026-01-15 05:04:31,623 - INFO - [AloeDAVClient._create] Collection created: http://vera-webdav-svc.vera.svc.cluster.local:5232/test/calendar-de3ce5e3-9b7a-49a4-996c-b21492a49b3f
http://radicale.org/ns/sync/e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
# =====================
# Insert Task
# =====================
task_filename: task-d2ceffc8-d05d-45a5-bf04-86ec2066ca07.ics

task_entry: uid='task-d2ceffc8-d05d-45a5-bf04-86ec2066ca07' content_type='text/calendar; charset=utf-8' file_ext='ics' version='2.0' rev=None prod_id='-//aloecraft.org//AloeDAV 1.0//EN' extended_attributes={} categories=[] etag='"33920d2bf99643572f03612785e26acd0edd444f90961d92b22eda4338c298e8"' raw_contents='BEGIN:VCALENDAR\r\nVERSION:2.0\r\nPRODID:-//aloecraft.org//AloeDAV 1.0//EN\r\nBEGIN:VTODO\r\nCLASS:PUBLIC\r\nDTSTAMP;X-VOBJ-FLOATINGTIME-ALLOWED=TRUE:20260115T050431\r\nDTSTART:20260110T120000\r\nPERCENT-COMPLETE:0\r\nPRIORITY:0\r\nSEQUENCE:0\r\nSTATUS:NEEDS-ACTION\r\nSUMMARY:Temporary Task\r

True